In [1]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense,
    AdditiveAttention, Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from rouge_score import rouge_scorer, scoring

2026-03-20 17:02:45.264024: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 17:02:46.107877: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 17:02:49.428648: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU enabled with memory growth")
    except RuntimeError as e:
        print(e)

Num GPUs Available: 1
GPU enabled with memory growth


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    text = re.sub("[" 
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  "]+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", " ", text)
    text = text.lower()
    text = " ".join(text.split())

    return text

In [4]:
df = pd.read_csv('../Data/news.tsv', sep='\t')

df["article"] = df["News body"].fillna("").apply(clean_text)
df["summary"] = df["Headline"].fillna("").apply(clean_text)

df = df[(df["article"].str.len() > 0) & (df["summary"].str.len() > 0)]

df["summary_in"] = "<sos> " + df["summary"]
df["summary_out"] = df["summary"] + " <eos>"

Tokenization

In [5]:
MAX_ART_LEN = 500
MAX_SUM_LEN = 50
VOCAB_SIZE = 30000

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<unk>",
    filters=""
)

tokenizer.fit_on_texts(
    df["article"].tolist() +
    df["summary_in"].tolist() +
    df["summary_out"].tolist()
)

encoder_input_seq = tokenizer.texts_to_sequences(df["article"])
decoder_input_seq = tokenizer.texts_to_sequences(df["summary_in"])
decoder_output_seq = tokenizer.texts_to_sequences(df["summary_out"])

encoder_input_seq = pad_sequences(encoder_input_seq, maxlen=MAX_ART_LEN, padding="post")
decoder_input_seq = pad_sequences(decoder_input_seq, maxlen=MAX_SUM_LEN, padding="post")
decoder_output_seq = pad_sequences(decoder_output_seq, maxlen=MAX_SUM_LEN, padding="post")

decoder_target_seq = decoder_output_seq[..., None]

vocab_size = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 30000


In [6]:
EMB_DIM = 128
LATENT_DIM = 256

encoder_inputs = Input(shape=(MAX_ART_LEN,))
encoder_embedding = Embedding(vocab_size, EMB_DIM, mask_zero=True)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(encoder_embedding)

decoder_inputs = Input(shape=(MAX_SUM_LEN,))
decoder_embedding = Embedding(vocab_size, EMB_DIM, mask_zero=True)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(decoder_embedding, initial_state=[state_h, state_c])

attention = AdditiveAttention()
context = attention([decoder_outputs, encoder_outputs])

decoder_concat = Concatenate(axis=-1)([decoder_outputs, context])

decoder_dense = Dense(vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1774026446.963157    3051 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20833 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:00:04.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 500, 128)  │  3,840,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 500)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 128)   │  3,840,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 500,      │    394,240 │ embedding[0][0],  │
│                     │ 256), (None,      │            │ not_equal[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 50, 256), │    394,240 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ lstm[0][1],       │
│                     │ (None, 256)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention  │ (None, 50, 256)   │        256 │ lstm_1[0][0],     │
│ (AdditiveAttention) │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 50, 512)   │          0 │ lstm_1[0][0],     │
│ (Concatenate)       │                   │            │ additive_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 30000) │ 15,390,000 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,858,736 (91.01 MB)

 Trainable params: 23,858,736 (91.01 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]

model.fit(
    [encoder_input_seq, decoder_input_seq],
    decoder_target_seq,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=callbacks
)

Epoch 1/20


2026-03-20 17:07:37.879173: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


1599/1599 ━━━━━━━━━━━━━━━━━━━━ 500s 308ms/step - accuracy: 0.1427 - loss: 6.7519 - val_accuracy: 0.1754 - val_loss: 6.0695
Epoch 2/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 493s 308ms/step - accuracy: 0.1997 - loss: 5.5982 - val_accuracy: 0.2137 - val_loss: 5.4283
Epoch 3/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 492s 308ms/step - accuracy: 0.2392 - loss: 4.9359 - val_accuracy: 0.2396 - val_loss: 5.1142
Epoch 4/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 492s 308ms/step - accuracy: 0.2729 - loss: 4.4408 - val_accuracy: 0.2552 - val_loss: 4.9557
Epoch 5/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 492s 308ms/step - accuracy: 0.3044 - loss: 4.0341 - val_accuracy: 0.2641 - val_loss: 4.8849
Epoch 6/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 492s 308ms/step - accuracy: 0.3366 - loss: 3.6849 - val_accuracy: 0.2697 - val_loss: 4.8720
Epoch 7/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 493s 308ms/step - accuracy: 0.3709 - loss: 3.3796 - val_accuracy: 0.2738 - val_loss: 4.8895
Epoch 8/20
1599/1599 ━━━━━━━━━━━━━━━━━━━━ 492s 308ms/step - accuracy: 0.4

In [8]:
model.save("seq2seq_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("config.json", "w") as f:
    json.dump({
        "MAX_ART_LEN": MAX_ART_LEN,
        "MAX_SUM_LEN": MAX_SUM_LEN,
        "VOCAB_SIZE": vocab_size
    }, f)

Inference

In [9]:
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

decoder_state_input_h = Input(shape=(LATENT_DIM,))
decoder_state_input_c = Input(shape=(LATENT_DIM,))
encoder_outputs_input = Input(shape=(None, LATENT_DIM))
decoder_input_token = Input(shape=(1,))

dec_emb = model.layers[3](decoder_input_token)
dec_outputs, h, c = model.layers[5](
    dec_emb,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

attn_out = model.layers[6]([dec_outputs, encoder_outputs_input])
dec_concat = model.layers[7]([dec_outputs, attn_out])
dec_outputs = model.layers[8](dec_concat)

decoder_model = Model(
    [decoder_input_token, encoder_outputs_input, decoder_state_input_h, decoder_state_input_c],
    [dec_outputs, h, c]
)

In [10]:
def generate_summary(text):
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    seq = pad_sequences(seq, maxlen=MAX_ART_LEN, padding="post")

    encoder_out, h, c = encoder_model.predict(seq, verbose=0)

    start_token = tokenizer.word_index.get("<sos>")
    end_token = tokenizer.word_index.get("<eos>")

    target_seq = np.array([[start_token]])
    decoded = []

    for _ in range(MAX_SUM_LEN):
        output, h, c = decoder_model.predict(
            [target_seq, encoder_out, h, c],
            verbose=0
        )

        idx = np.argmax(output[0, -1, :])
        word = tokenizer.index_word.get(idx, "")

        if word == "<eos>" or word == "":
            break

        decoded.append(word)
        target_seq = np.array([[idx]])

    return " ".join(decoded)

In [11]:
print("\nARTICLE:\n", df["article"].iloc[0][:300])
print("\nREFERENCE:\n", df["summary"].iloc[0])
print("\nPREDICTED:\n", generate_summary(df["article"].iloc[0]))


ARTICLE:
 only five internationals allowed, count em, five! so first off we should say, per our usual atlanta united lineup predictions, this will be wrong. why will it be wrong? well, aside from the obvious, we still don t have a ton of data points from frank de boer in how he prefers to rotate his team for 

REFERENCE:
 predicting atlanta united s lineup against columbus crew in the u.s. open cup

PREDICTED:
 orlando city preview how to watch the united states


In [12]:
MAX_TEST = min(100, len(df))

preds = []
refs = df["summary"].iloc[:MAX_TEST].tolist()

for i in tqdm(range(MAX_TEST)):
    preds.append(generate_summary(df["article"].iloc[i]))

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL', 'rougeLsum'],
    use_stemmer=True
)

aggregator = scoring.BootstrapAggregator()

for r, p in zip(refs, preds):
    aggregator.add_scores(scorer.score(r, p))

res = aggregator.aggregate()

print("\nROUGE SCORES:")
print("rouge1:", res['rouge1'].mid.fmeasure)
print("rouge2:", res['rouge2'].mid.fmeasure)
print("rougeL:", res['rougeL'].mid.fmeasure)
print("rougeLsum:", res['rougeLsum'].mid.fmeasure)

100%|██████████| 100/100 [01:46<00:00,  1.07s/it]



ROUGE SCORES:
rouge1: 0.14479029968616305
rouge2: 0.04059512632863016
rougeL: 0.13430206448473536
rougeLsum: 0.13389774147521877


In [13]:
OUTPUT_CSV = "summarization_comparison_results.csv"

new_results = pd.DataFrame([{
    "Model": "Seq2Seq LSTM",
    "rouge1": round(res['rouge1'].mid.fmeasure, 4),
    "rouge2": round(res['rouge2'].mid.fmeasure, 4),
    "rougeL": round(res['rougeL'].mid.fmeasure, 4),
    "rougeLsum": round(res['rougeLsum'].mid.fmeasure, 4),
    "Average Score": round(np.mean([
        res['rouge1'].mid.fmeasure,
        res['rouge2'].mid.fmeasure,
        res['rougeL'].mid.fmeasure,
        res['rougeLsum'].mid.fmeasure
    ]), 4)
}])

if os.path.exists(OUTPUT_CSV):
    old_df = pd.read_csv(OUTPUT_CSV)
    final_df = pd.concat([old_df, new_results], ignore_index=True)
else:
    final_df = new_results

final_df = final_df.drop_duplicates(subset=["Model"], keep="last")
final_df = final_df.sort_values(by="Average Score", ascending=False)

final_df.to_csv(OUTPUT_CSV, index=False)

print("\nComparison of all Models based on F1 score")
print(final_df)


Comparison of all Models based on F1 score
          Model  rouge1  rouge2  rougeL  rougeLsum  Average Score
0      TextRank  0.5028  0.3924  0.4402     0.4402         0.4451
1        TF-IDF  0.4836  0.3779  0.4144     0.4144         0.4253
2  Seq2Seq LSTM  0.1448  0.0406  0.1343     0.1339         0.1134


Additional artifacts for UI requirement

In [1]:
import json
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, AdditiveAttention, Concatenate
from tensorflow.keras.models import Model

2026-03-22 07:50:27.943255: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
MODEL_PATH = "seq2seq_model.h5"
TOKENIZER_PATH = "tokenizer.pkl"
CONFIG_PATH = "config.json"

SAVE_DIR = "./artifacts/DL"

In [3]:
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

with open(CONFIG_PATH) as f:
    config = json.load(f)

MAX_ART_LEN = config["MAX_ART_LEN"]
MAX_SUM_LEN = config["MAX_SUM_LEN"]
VOCAB_SIZE = config["VOCAB_SIZE"]

EMB_DIM = 128
LATENT_DIM = 256

In [4]:
print("Rebuilding model architecture...")

# Encoder
encoder_inputs = Input(shape=(MAX_ART_LEN,))
encoder_embedding = Embedding(VOCAB_SIZE, EMB_DIM, mask_zero=True)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(MAX_SUM_LEN,))
decoder_embedding = Embedding(VOCAB_SIZE, EMB_DIM, mask_zero=True)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True
)(decoder_embedding, initial_state=[state_h, state_c])

# Attention
attention = AdditiveAttention()
context = attention([decoder_outputs, encoder_outputs])

decoder_concat = Concatenate(axis=-1)([decoder_outputs, context])

decoder_dense = Dense(VOCAB_SIZE, activation="softmax")
decoder_outputs = decoder_dense(decoder_concat)

# Full model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

Rebuilding model architecture...


2026-03-22 07:52:47.788850: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-03-22 07:52:48.380463: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 61440000 exceeds 10% of free system memory.
2026-03-22 07:52:48.502442: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 61440000 exceeds 10% of free system memory.
2026-03-22 07:52:48.545242: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 61440000 exceeds 10% of free system memory.


In [5]:
model.load_weights(MODEL_PATH)
print("Weights loaded successfully!")

Weights loaded successfully!


2026-03-22 07:53:22.659562: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 61440000 exceeds 10% of free system memory.


In [6]:
print("Building encoder model...")

encoder_model = Model(
    encoder_inputs,
    [encoder_outputs, state_h, state_c]
)

Building encoder model...


In [7]:
print("Building decoder model...")

decoder_state_input_h = Input(shape=(LATENT_DIM,))
decoder_state_input_c = Input(shape=(LATENT_DIM,))
encoder_outputs_input = Input(shape=(MAX_ART_LEN, LATENT_DIM))
decoder_input_token = Input(shape=(1,))

dec_emb_layer = model.layers[3]
dec_emb = dec_emb_layer(decoder_input_token)


dec_lstm_layer = model.layers[5]
dec_outputs, h, c = dec_lstm_layer(
    dec_emb,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)


attn_layer = model.layers[6]
attn_out = attn_layer([dec_outputs, encoder_outputs_input])


concat_layer = model.layers[7]
dec_concat = concat_layer([dec_outputs, attn_out])


dense_layer = model.layers[8]
dec_outputs = dense_layer(dec_concat)


decoder_model = Model(
    [decoder_input_token, encoder_outputs_input, decoder_state_input_h, decoder_state_input_c],
    [dec_outputs, h, c]
)

Building decoder model...


In [8]:
encoder_model.save(f"{SAVE_DIR}/encoder_model.keras")
decoder_model.save(f"{SAVE_DIR}/decoder_model.keras")

print(" Models saved successfully.")

 Models saved successfully.
